# SalesInsight PY

*Análise e visualização de dados de vendas*

**Autor:** Bruno Miguel Corrêa

**Curso:** Desenvolvimento de IA para Análise Preditiva — SENAI/SC


## Visão Geral

O SalesInsight PY estrutura um fluxo completo de análise de dados de vendas, desde a inspeção da base bruta até a geração de métricas, segmentações e visualizações.

O processamento utiliza Python, Pandas e NumPy. Os resultados gráficos foram produzidos com Matplotlib e Seaborn.


## 1. Configuração do Ambiente

Importação das dependências e identificação das versões utilizadas na execução.


In [50]:
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sales_insight.dicionario_dados import DICIONARIO_DADOS

print("=== Ambiente de execução ===")
print(f"Python: {platform.python_version()}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

=== Ambiente de execução ===
Python: 3.13.14
Pandas: 3.0.5
NumPy:  2.5.1


## 2. Carregamento e Inspeção dos Dados

### `RF01` — Carregamento do Dataset

O `vendas.csv` tem como base o [E-commerce Analytics Dataset Brazil](https://www.kaggle.com/datasets/joocarlosjr/e-commerce-analytics-dataset-brazil). Os dados foram consolidados, adaptados e enriquecidos para representar informações comerciais e logísticas.

Também foram introduzidas inconsistências controladas para viabilizar as etapas de limpeza e validação. O arquivo bruto permanece preservado em `data/raw`.


In [51]:
caminho_dados = Path("../data/raw/vendas.csv")

df_bruto = pd.read_csv(caminho_dados)
df = df_bruto.copy()

### `RF02` — Inspeção Estrutural

A inspeção inicial apresenta uma amostra dos registros, o dicionário de dados, as dimensões da base, os tipos das colunas e a ocorrência de valores ausentes.


In [52]:
display(df.head(3).style.hide(axis="index"))

id_venda,data_venda,id_cliente,nome_cliente,cidade,estado,regiao,id_produto,produto,categoria,quantidade,preco_unitario,desconto,previsao_entrega,data_entrega
ORD00001,2025-01-01,Cliente_008,Thales Pereira,Cascavel,PR,Sul,P0005,"Fone de Ouvido Sem Fio TWS, PHILIPS",Áudio,1.000000,124.910000,0.032700,2025-01-07,2025-01-09
ORD00002,2025-01-01,Cliente_011,Julia Ribeiro,Santa Maria,RS,Sul,P0006,Fone de ouvido Sem Fio QCY T27,Áudio,4.000000,129.860000,0.047700,2025-01-04,2025-01-08
ORD00003,2025-01-01,Cliente_019,Gabriela da Paz,Uberlândia,MG,Sudeste,P0002,Samsung Galaxy Tab S6 Lite,Smartphones e Tablets,2.000000,1799.010000,0.148100,2025-01-05,2025-01-04


#### Dicionário dos Dados


In [53]:
display(pd.DataFrame(DICIONARIO_DADOS).style.hide(axis="index"))

Coluna,Descrição
id_venda,Identificador único da venda.
data_venda,Data em que a venda foi realizada.
id_cliente,Identificador do cliente associado à venda.
nome_cliente,Nome do cliente.
cidade,Cidade de residência do cliente.
estado,Estado de residência do cliente.
regiao,Região geográfica associada ao cliente.
id_produto,Identificador do produto vendido.
produto,Nome do produto vendido.
categoria,Categoria à qual o produto pertence.


#### Dimensões, Tipos e Valores Ausentes


In [55]:
def resumo_estrutural(dataframe):
    """
    Retorna os tipos e valores ausentes das colunas.
    """
    return pd.DataFrame(
        {
            "coluna": dataframe.columns,
            "tipo": dataframe.dtypes.astype(str).values,
            "valores_ausentes": dataframe.isna().sum().values,
            "percentual_ausente": dataframe.isna().mean().values * 100,
        }
    )


linhas, colunas = df.shape
resumo_inicial = resumo_estrutural(df)

print(f"Dimensões: {linhas} linhas × {colunas} colunas")

display(
    resumo_inicial.style.hide(axis="index").format({"percentual_ausente": "{:.2f}%"})
)

Dimensões: 5732 linhas × 15 colunas


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,str,63,1.10%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


#### Diagnóstico Inicial

A base bruta contém 5.732 registros e 15 colunas. As colunas `data_venda`, `previsao_entrega` e `data_entrega` foram carregadas como texto e precisam ser convertidas para `datetime`.

Foram identificados valores ausentes em `data_venda` (63), `quantidade` (229) e `preco_unitario` (86). Esses pontos definem o escopo da limpeza realizada no RF03.


## 3. Limpeza e Tratamento dos Dados

### `RF03` — Padronização e Validação

A limpeza contempla a normalização de identificadores e textos, conversão das datas, validação das variáveis numéricas e remoção de registros incompletos nas colunas críticas.


In [56]:
registros_iniciais = len(df)

padroes_identificadores = {
    "id_venda": re.compile(r"^ORD\d{5}$"),
    "id_cliente": re.compile(r"^Cliente_\d{3}$"),
    "id_produto": re.compile(r"^P\d{4}$"),
}


def validar_identificadores(dataframe, padroes):
    """
    Retorna o resultado da validação dos identificadores.
    """
    resultados = []

    for coluna, padrao in padroes.items():
        validos = dataframe[coluna].astype("string").str.fullmatch(padrao, na=False)

        resultados.append(
            {
                "coluna": coluna,
                "valores_ausentes": int(dataframe[coluna].isna().sum()),
                "fora_do_padrao": int((~validos).sum()),
                "duplicados": (
                    int(dataframe[coluna].duplicated().sum())
                    if coluna == "id_venda"
                    else pd.NA
                ),
            }
        )

    return pd.DataFrame(resultados)


validacao_ids_antes = validar_identificadores(
    df,
    padroes_identificadores,
)

ids_clientes_corrigidos = int(
    validacao_ids_antes.loc[
        validacao_ids_antes["coluna"] == "id_cliente",
        "fora_do_padrao",
    ].iloc[0]
)

display(validacao_ids_antes.style.hide(axis="index"))

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,226,
id_produto,0,0,


#### 3.1 Padronização dos Identificadores e Textos

A validação identificou 226 ocorrências de `id_cliente` fora do padrão esperado. Esses valores serão normalizados com expressão regular. As demais colunas textuais terão espaços excedentes removidos.


In [57]:
def normalizar_id_cliente(valor):
    """Padroniza o identificador para o formato Cliente_000."""
    return re.sub(
        r"^cliente\D*(\d{3})\D*$",
        r"Cliente_\1",
        str(valor).strip(),
        flags=re.IGNORECASE,
    )


df["id_cliente"] = df["id_cliente"].apply(normalizar_id_cliente)

colunas_textuais = [
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "produto",
    "categoria",
]

resultado_padronizacao = []

for coluna in colunas_textuais:
    original = df[coluna].copy()
    padronizado = original.str.strip()

    resultado_padronizacao.append(
        {
            "coluna": coluna,
            "unicos_antes": original.nunique(),
            "unicos_depois": padronizado.nunique(),
            "espacos_corrigidos": int(original.ne(padronizado).sum()),
        }
    )

    df[coluna] = padronizado

resumo_padronizacao = pd.DataFrame(resultado_padronizacao)

validacao_ids_depois = validar_identificadores(
    df,
    padroes_identificadores,
)

display(validacao_ids_depois.style.hide(axis="index"))

display(resumo_padronizacao.style.hide(axis="index"))

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,0,
id_produto,0,0,


coluna,unicos_antes,unicos_depois,espacos_corrigidos
nome_cliente,530,530,0
cidade,105,105,0
estado,22,22,0
regiao,4,4,0
produto,72,27,103
categoria,15,5,69


#### 3.2 Conversão das Datas

As colunas temporais serão convertidas para `datetime`. O parâmetro `errors="coerce"` transforma valores incompatíveis em `NaT`, permitindo identificar falhas de conversão.


In [58]:
colunas_datas = [
    "data_venda",
    "previsao_entrega",
    "data_entrega",
]

datas_originais = df[colunas_datas].copy()

df[colunas_datas] = df[colunas_datas].apply(
    pd.to_datetime,
    errors="coerce",
)

falhas_conversao = (df[colunas_datas].isna() & datas_originais.notna()).sum()

validacao_datas = pd.DataFrame(
    {
        "coluna": colunas_datas,
        "tipo_final": [str(df[coluna].dtype) for coluna in colunas_datas],
        "falhas_conversao": falhas_conversao.values,
    }
)

display(validacao_datas.style.hide(axis="index"))

coluna,tipo_final,falhas_conversao
data_venda,datetime64[us],0
previsao_entrega,datetime64[us],0
data_entrega,datetime64[us],0


#### 3.3 Tratamento das Ausências e Validação Numérica

Os registros com ausência em `data_venda`, `quantidade` ou `preco_unitario` serão removidos. Como existem linhas com ausência em mais de uma dessas colunas, a remoção considera os registros únicos afetados.


In [59]:
colunas_criticas = [
    "data_venda",
    "quantidade",
    "preco_unitario",
]

ausencias_criticas = (
    df[colunas_criticas]
    .isna()
    .sum()
    .rename_axis("coluna")
    .reset_index(name="valores_ausentes")
)

mascara_remocao = df[colunas_criticas].isna().any(axis=1)

total_removidos = int(mascara_remocao.sum())

df = df.loc[~mascara_remocao].copy()

display(ausencias_criticas.style.hide(axis="index"))

print(f"Registros únicos removidos: {total_removidos}")

coluna,valores_ausentes
data_venda,63
quantidade,229
preco_unitario,86


Registros únicos removidos: 371


In [60]:
validacoes_numericas = {
    "quantidade_inteira": bool((df["quantidade"] % 1 == 0).all()),
    "quantidade_positiva": bool((df["quantidade"] > 0).all()),
    "preco_positivo": bool((df["preco_unitario"] > 0).all()),
    "desconto_valido": bool(df["desconto"].between(0, 1).all()),
}

if not all(validacoes_numericas.values()):
    raise ValueError("Foram encontrados valores numéricos inválidos.")

df["quantidade"] = df["quantidade"].astype("int64")

colunas_numericas = [
    "quantidade",
    "preco_unitario",
    "desconto",
]

validacao_numerica = pd.DataFrame(
    {
        "coluna": colunas_numericas,
        "tipo": (df[colunas_numericas].dtypes.astype(str).values),
        "valor_minimo": [df[coluna].min() for coluna in colunas_numericas],
        "valor_maximo": [df[coluna].max() for coluna in colunas_numericas],
        "regra_validada": [
            (
                validacoes_numericas["quantidade_inteira"]
                and validacoes_numericas["quantidade_positiva"]
            ),
            validacoes_numericas["preco_positivo"],
            validacoes_numericas["desconto_valido"],
        ],
    }
)

display(validacao_numerica.style.hide(axis="index"))

coluna,tipo,valor_minimo,valor_maximo,regra_validada
quantidade,int64,1.000000,4.000000,True
preco_unitario,float64,17.900000,4604.000000,True
desconto,float64,0.000100,0.350000,True


#### Resultado da Limpeza


In [61]:
registros_finais = len(df)

espacos_corrigidos = {
    linha.coluna: linha.espacos_corrigidos for linha in resumo_padronizacao.itertuples()
}

relatorio_limpeza = {
    "registros_iniciais": registros_iniciais,
    "ids_cliente_normalizados": ids_clientes_corrigidos,
    "espacos_produto_corrigidos": espacos_corrigidos["produto"],
    "espacos_categoria_corrigidos": espacos_corrigidos["categoria"],
    "registros_removidos": total_removidos,
    "registros_finais": registros_finais,
}

tabela_relatorio_limpeza = pd.DataFrame(
    relatorio_limpeza.items(),
    columns=["indicador", "valor"],
)

display(tabela_relatorio_limpeza.style.hide(axis="index"))

display(
    resumo_estrutural(df)
    .style.hide(axis="index")
    .format({"percentual_ausente": "{:.2f}%"})
)

indicador,valor
registros_iniciais,5732
ids_cliente_normalizados,226
espacos_produto_corrigidos,103
espacos_categoria_corrigidos,69
registros_removidos,371
registros_finais,5361


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,datetime64[us],0,0.00%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


Foram normalizados 226 identificadores de clientes, 103 ocorrências com espaços excedentes em produtos e 69 em categorias. As datas foram convertidas sem falhas de formato.

A remoção das ausências críticas eliminou 371 registros únicos, mantendo 5.361 vendas válidas para as etapas seguintes.


## 4. Transformação e Criação de Variáveis

### `RF04` — Colunas Derivadas e Transformações Condicionais

A base limpa será ampliada com variáveis financeiras, temporais, logísticas e categóricas. As transformações são aplicadas diretamente sobre as colunas do DataFrame.


#### 4.1 Variáveis Financeiras

A receita representa o valor líquido da venda. O desconto é aplicado ao preço unitário, arredondado em centavos e multiplicado pela quantidade vendida.

As transações também serão classificadas como `Baixo Valor`, `Médio Valor` ou `Alto Valor` por meio de `np.select`.


In [62]:
preco_liquido_unitario = (df["preco_unitario"] * (1 - df["desconto"])).round(2)

df["receita_total"] = (df["quantidade"] * preco_liquido_unitario).round(2)

df["valor_desconto"] = (
    df["quantidade"] * df["preco_unitario"] - df["receita_total"]
).round(2)

condicoes_receita = [
    df["receita_total"] < 500,
    df["receita_total"].between(500, 4999.99),
    df["receita_total"] >= 5000,
]

faixas_receita = [
    "Baixo Valor",
    "Médio Valor",
    "Alto Valor",
]

df["faixa_receita_item"] = np.select(
    condicoes_receita,
    faixas_receita,
    default="Não Classificado",
)

display(
    df[
        [
            "quantidade",
            "preco_unitario",
            "desconto",
            "valor_desconto",
            "receita_total",
            "faixa_receita_item",
        ]
    ]
    .head(5)
    .style.hide(axis="index")
    .format(
        {
            "preco_unitario": "R$ {:,.2f}",
            "desconto": "{:.2%}",
            "valor_desconto": "R$ {:,.2f}",
            "receita_total": "R$ {:,.2f}",
        }
    )
)

quantidade,preco_unitario,desconto,valor_desconto,receita_total,faixa_receita_item
1,R$ 124.91,3.27%,R$ 4.08,R$ 120.83,Baixo Valor
4,R$ 129.86,4.77%,R$ 24.76,R$ 494.68,Baixo Valor
2,"R$ 1,799.01",14.81%,R$ 532.86,"R$ 3,065.16",Médio Valor
4,R$ 53.90,14.02%,R$ 30.24,R$ 185.36,Baixo Valor
2,R$ 389.90,11.26%,R$ 87.80,R$ 692.00,Médio Valor


#### 4.2 Variáveis Temporais

A data da venda será decomposta em mês numérico, nome do mês, trimestre e ano. O nome do mês será obtido por meio de um dicionário em português.


In [64]:
meses = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho"}

df["mes"] = df["data_venda"].dt.month
df["mes_venda"] = df["mes"].map(meses)
df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
df["ano"] = df["data_venda"].dt.year

periodos_identificados = (
    df[
        [
            "ano",
            "trimestre",
            "mes",
            "mes_venda",
        ]
    ]
    .drop_duplicates()
    .sort_values(["ano", "mes"])
    .reset_index(drop=True)
)

print(
    "Período analisado: "
    f"{df['data_venda'].min():%d/%m/%Y} a "
    f"{df['data_venda'].max():%d/%m/%Y}"
)

display(periodos_identificados.style.hide(axis="index"))

Período analisado: 01/01/2025 a 30/06/2025


ano,trimestre,mes,mes_venda
2025,Q1,1,Janeiro
2025,Q1,2,Fevereiro
2025,Q1,3,Março
2025,Q2,4,Abril
2025,Q2,5,Maio
2025,Q2,6,Junho


#### 4.3 Variáveis Logísticas

O desvio entre a data efetiva e a previsão de entrega será calculado em dias. Valores positivos representam atrasos; valores negativos representam entregas antecipadas.


In [65]:
df["desvio_entrega_dias"] = (df["data_entrega"] - df["previsao_entrega"]).dt.days

df["atrasado"] = df["desvio_entrega_dias"] > 0

resumo_entregas = pd.DataFrame(
    {
        "situacao": [
            "Antecipada",
            "No prazo",
            "Atrasada",
        ],
        "numero_entregas": [
            int((df["desvio_entrega_dias"] < 0).sum()),
            int((df["desvio_entrega_dias"] == 0).sum()),
            int(df["atrasado"].sum()),
        ],
    }
)

display(resumo_entregas.style.hide(axis="index"))

situacao,numero_entregas
Antecipada,2364
No prazo,1693
Atrasada,1304


#### 4.4 Dataset Processado

As colunas serão organizadas por domínio e o resultado será salvo em `data/processed/vendas_processado.csv`.


In [67]:
ordem_colunas = [
    "id_venda",
    "data_venda",
    "ano",
    "trimestre",
    "mes",
    "mes_venda",
    "id_cliente",
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "id_produto",
    "produto",
    "categoria",
    "quantidade",
    "preco_unitario",
    "desconto",
    "valor_desconto",
    "receita_total",
    "faixa_receita_item",
    "previsao_entrega",
    "data_entrega",
    "desvio_entrega_dias",
    "atrasado",
]

df = df[ordem_colunas]

pasta_processados = Path("../data/processed")
pasta_processados.mkdir(
    parents=True,
    exist_ok=True,
)

caminho_saida = pasta_processados / "vendas_processado.csv"

df.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig",
)

print(f"Dataset processado: " f"{df.shape[0]} registros × " f"{df.shape[1]} colunas")

print(f"Arquivo salvo em: {caminho_saida}")

Dataset processado: 5361 registros × 24 colunas
Arquivo salvo em: ../data/processed/vendas_processado.csv


Foram criadas nove variáveis derivadas: três financeiras, quatro temporais e duas logísticas. O dataset processado contém 5.361 registros e 24 colunas, mantendo os resultados financeiros adotados nas etapas anteriores.


## 5. Métricas Agregadas de Vendas

### `RF05` — Agregações com `groupby`

As vendas serão consolidadas por período, produto, categoria e região. As quatro métricas serão retornadas em um dicionário de DataFrames.


In [68]:
def calcular_metricas(dataframe):
    """Calcula e retorna as métricas agregadas de vendas."""
    por_mes = (
        dataframe.groupby(
            ["mes", "mes_venda"],
            as_index=False,
        )
        .agg(
            receita_total=("receita_total", "sum"),
            unidades_vendidas=("quantidade", "sum"),
            numero_vendas=("id_venda", "nunique"),
        )
        .sort_values("mes")
        .reset_index(drop=True)
    )

    por_produto = (
        dataframe.groupby("produto", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_categoria = (
        dataframe.groupby("categoria", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_regiao = (
        dataframe.groupby("regiao", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
            numero_vendas=("id_venda", "nunique"),
            ticket_medio=("receita_total", "mean"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_regiao["ticket_medio"] = por_regiao["ticket_medio"].round(2)

    return {
        "por_mes": por_mes,
        "por_produto": por_produto,
        "por_categoria": por_categoria,
        "por_regiao": por_regiao,
    }


metricas_gerais = calcular_metricas(df)

#### 5.1 Desempenho Mensal

Como a receita, a quantidade vendida e o número de vendas se distribuíram entre os meses?


In [69]:
display(
    metricas_gerais["por_mes"]
    .style.hide(axis="index")
    .format(
        {
            "receita_total": "R$ {:,.2f}",
        }
    )
)

mes,mes_venda,receita_total,unidades_vendidas,numero_vendas
1,Janeiro,"R$ 1,299,266.31",1968,824
2,Fevereiro,"R$ 1,591,691.83",2063,855
3,Março,"R$ 2,291,715.46",2443,991
4,Abril,"R$ 2,004,674.59",2241,900
5,Maio,"R$ 1,613,016.68",2081,880
6,Junho,"R$ 1,537,652.30",2131,911


#### 5.2 Produtos com Maior Receita

Quais produtos apresentaram as cinco maiores receitas acumuladas?


In [70]:
top_5_produtos = metricas_gerais["por_produto"].head(5)

display(
    top_5_produtos.style.hide(axis="index").format(
        {
            "receita_total": "R$ {:,.2f}",
        }
    )
)

produto,receita_total
ACER Notebook Gamer Nitro,"R$ 1,848,972.94"
"Projetor Smart Epson EpiqVision, FULL HD","R$ 1,244,636.36"
Samsung Galaxy A36,"R$ 850,244.53"
Samsung Galaxy Tab S6 Lite,"R$ 839,934.77"
Projetor EPSON Powerlite Wide Screen,"R$ 801,184.04"


#### 5.3 Receita por Categoria

Como a receita total se distribuiu entre as categorias de produtos?


In [71]:
display(
    metricas_gerais["por_categoria"]
    .style.hide(axis="index")
    .format(
        {
            "receita_total": "R$ {:,.2f}",
        }
    )
)

categoria,receita_total
Informática,"R$ 3,049,354.74"
TV e Projeção,"R$ 3,022,977.33"
Áudio,"R$ 2,140,350.33"
Smartphones e Tablets,"R$ 2,017,909.85"
Acessórios Mobile,"R$ 107,424.92"


#### 5.4 Desempenho Regional

Como as regiões se comparam em receita total, número de vendas e ticket médio?


In [72]:
display(
    metricas_gerais["por_regiao"]
    .style.hide(axis="index")
    .format(
        {
            "receita_total": "R$ {:,.2f}",
            "ticket_medio": "R$ {:,.2f}",
        }
    )
)

regiao,receita_total,numero_vendas,ticket_medio
Sudeste,"R$ 2,829,006.77",1435,"R$ 1,971.43"
Sul,"R$ 2,776,130.94",1477,"R$ 1,879.57"
Norte,"R$ 2,393,107.34",1232,"R$ 1,942.46"
Nordeste,"R$ 2,339,772.12",1217,"R$ 1,922.57"


Março apresentou o maior desempenho mensal, com receita de R$ 2,29 milhões, 2.443 unidades vendidas e 991 vendas.

O ACER Notebook Gamer Nitro liderou o ranking de produtos, com R$ 1,85 milhão. Informática foi a categoria com maior receita, seguida de TV e Projeção.

O Sudeste registrou a maior receita regional e o maior ticket médio. O Sul apresentou o maior número de vendas, mas o menor ticket médio entre as regiões.


## 6. Segmentação de Clientes

### `RF06` — Classificação por Nível de Gasto

As vendas serão agrupadas por cliente e classificadas de acordo com a receita acumulada.

| Gasto acumulado               | Segmento |
| ----------------------------- | -------- |
| Abaixo de R\$ 5.000,00         | Bronze   |
| De R\$ 5.000,00 a R\$ 15.000,00 | Prata    |
| Acima de R\$ 15.000,00         | Ouro     |


In [73]:
def segmentar_clientes(dataframe):
    """Agrupa e classifica os clientes pelo gasto acumulado."""
    clientes = dataframe.groupby(
        ["id_cliente", "nome_cliente"],
        as_index=False,
    ).agg(
        gasto_total=("receita_total", "sum"),
    )

    clientes["gasto_total"] = clientes["gasto_total"].round(2)

    clientes["segmento"] = clientes["gasto_total"].apply(
        lambda gasto: (
            "Bronze" if gasto < 5_000 else "Prata" if gasto <= 15_000 else "Ouro"
        )
    )

    return clientes.sort_values(
        "gasto_total",
        ascending=False,
    ).reset_index(drop=True)


clientes_segmentados = segmentar_clientes(df)

#### 6.1 Distribuição por Segmento


In [74]:
ordem_segmentos = [
    "Ouro",
    "Prata",
    "Bronze",
]

distribuicao_segmentos = (
    clientes_segmentados["segmento"]
    .value_counts()
    .reindex(
        ordem_segmentos,
        fill_value=0,
    )
    .rename_axis("segmento")
    .reset_index(name="numero_clientes")
)

distribuicao_segmentos["percentual"] = (
    distribuicao_segmentos["numero_clientes"] / len(clientes_segmentados) * 100
).round(2)

display(
    distribuicao_segmentos.style.hide(axis="index").format(
        {
            "percentual": "{:.2f}%",
        }
    )
)

segmento,numero_clientes,percentual
Ouro,245,45.88%
Prata,182,34.08%
Bronze,107,20.04%


#### 6.2 Clientes com Maior Gasto

A tabela apresenta os dez clientes com maior receita acumulada no período.


In [75]:
top_10_clientes = clientes_segmentados.head(10)

display(
    top_10_clientes.style.hide(axis="index").format(
        {
            "gasto_total": "R$ {:,.2f}",
        }
    )
)

id_cliente,nome_cliente,gasto_total,segmento
Cliente_371,Ana Luiza Nunes,"R$ 116,525.85",Ouro
Cliente_165,Maria Fernanda Teixeira,"R$ 111,976.65",Ouro
Cliente_394,Eduardo Oliveira,"R$ 108,436.37",Ouro
Cliente_008,Thales Pereira,"R$ 96,223.86",Ouro
Cliente_360,Mariane Campos,"R$ 92,852.62",Ouro
Cliente_300,Dra. Maria Sophia Nogueira,"R$ 89,455.87",Ouro
Cliente_218,Isabel Pereira,"R$ 89,303.07",Ouro
Cliente_337,Benício Fernandes,"R$ 87,821.42",Ouro
Cliente_266,Caroline Carvalho,"R$ 87,502.24",Ouro
Cliente_357,Amanda Araújo,"R$ 81,303.37",Ouro


Foram classificados 534 clientes. O segmento Ouro concentra 245 clientes (45,88%), seguido por Prata, com 182, e Bronze, com 107.

Ana Luiza Nunes apresentou o maior gasto acumulado, com R$ 116.525,85. Todos os clientes do Top 10 pertencem ao segmento Ouro.


## 7. Operações Numéricas com NumPy

### `RF07` — Vetorização, Broadcasting e Filtragem

A receita será convertida de `Series` para um array NumPy. Sobre esse array serão aplicadas funções de agregação, operações vetorizadas com escalares e filtragem por máscara booleana.


In [76]:
receitas_array = df["receita_total"].to_numpy()


def calcular_estatisticas_numpy(valores):
    """Calcula estatísticas agregadas sobre um array NumPy."""
    return {
        "media": float(np.mean(valores)),
        "mediana": float(np.median(valores)),
        "desvio_padrao": float(np.std(valores)),
        "soma": float(np.sum(valores)),
        "minimo": float(np.min(valores)),
        "maximo": float(np.max(valores)),
    }


estatisticas_receita = calcular_estatisticas_numpy(receitas_array)

resumo_array = pd.DataFrame(
    [
        {
            "tipo": str(receitas_array.dtype),
            "dimensoes": receitas_array.ndim,
            "elementos": receitas_array.size,
        }
    ]
)

tabela_estatisticas = pd.DataFrame(
    estatisticas_receita.items(),
    columns=["medida", "valor"],
)

display(resumo_array.style.hide(axis="index"))

display(
    tabela_estatisticas.style.hide(axis="index").format(
        {
            "valor": "R$ {:,.2f}",
        }
    )
)

tipo,dimensoes,elementos
float64,1,5361


medida,valor
media,"R$ 1,928.37"
mediana,R$ 979.70
desvio_padrao,"R$ 2,530.21"
soma,"R$ 10,338,017.17"
minimo,R$ 13.83
maximo,"R$ 18,316.56"


#### 7.1 Escalonamento Vetorizado

O array será escalonado para o intervalo entre 0 e 1. A subtração e a divisão por valores escalares demonstram broadcasting, sem utilização de laços.


In [77]:
valor_minimo = estatisticas_receita["minimo"]
valor_maximo = estatisticas_receita["maximo"]

if valor_maximo == valor_minimo:
    raise ValueError("Não é possível escalonar um array sem variação.")

receitas_escalonadas = (receitas_array - valor_minimo) / (valor_maximo - valor_minimo)

print(
    "Intervalo após o escalonamento: "
    f"{receitas_escalonadas.min():.1f} a "
    f"{receitas_escalonadas.max():.1f}"
)

Intervalo após o escalonamento: 0.0 a 1.0


#### 7.2 Filtragem Acima da Média

Uma máscara booleana será utilizada para selecionar as vendas cuja receita supera a média do período.


In [78]:
media_receita = estatisticas_receita["media"]

mascara_acima_media = receitas_array > media_receita

receitas_acima_media = receitas_array[mascara_acima_media]

quantidade_vendas = receitas_array.size
quantidade_acima_media = receitas_acima_media.size

percentual_acima_media = quantidade_acima_media / quantidade_vendas * 100

resumo_filtro = pd.DataFrame(
    [
        {
            "total_vendas": quantidade_vendas,
            "vendas_acima_media": quantidade_acima_media,
            "percentual": percentual_acima_media,
        }
    ]
)

display(
    resumo_filtro.style.hide(axis="index").format(
        {
            "percentual": "{:.2f}%",
        }
    )
)

total_vendas,vendas_acima_media,percentual
5361,1720,32.08%


O array contém 5.361 receitas do tipo `float64`. A receita média foi de R\$ 1.928,37 e a mediana de R\$ 979,70. O desvio-padrão populacional, calculado por `np.std()` com `ddof=0`, foi de R\$ 2.530,21.

O escalonamento confirmou o intervalo entre 0 e 1. A máscara booleana identificou 1.720 vendas acima da média, equivalentes a 32,08% das transações.


## 8. Visualização dos Resultados

### `RF08` — Matplotlib e Seaborn

As figuras foram produzidas no notebook de visualizações e exportadas para `reports/figures`. Esta seção reúne os resultados gráficos e suas interpretações.


#### 8.1 Evolução da Receita Mensal

Como a receita se comportou ao longo do período analisado?

![Receita total por mês](../reports/figures/receita_por_mes.png)

A receita aumentou de $\text{R\$}$ 1,30 milhão em janeiro para $\text{R\$}$ 2,29 milhões em março, maior resultado do período. Após o pico, recuou continuamente até atingir $\text{R\$}$ 1,54 milhão em junho.


#### 8.2 Produtos com Maior Receita

Quais produtos apresentaram as maiores receitas acumuladas?

![Produtos com maior receita](../reports/figures/top_produtos.png)

O ACER Notebook Gamer Nitro liderou o ranking, com $\text{R\$}$ 1,85 milhão, seguido pelo Projetor Smart Epson EpiqVision, com $\text{R\$}$ 1,24 milhão. Os demais produtos do Top 5 permaneceram abaixo de $\text{R\$}$ 860 mil.


#### 8.3 Quantidade Vendida e Receita

A quantidade vendida determina o valor da transação?

![Relação entre quantidade e receita](../reports/figures/quantidade_vs_receita.png)

Transações com a mesma quantidade apresentam receitas bastante diferentes. Portanto, a quantidade não explica isoladamente o valor da venda; o preço e a categoria dos produtos também influenciam diretamente o resultado.


#### 8.4 Desempenho Comercial das Cidades

Quais cidades concentram o maior número de vendas e as maiores receitas?

![Cidades com mais vendas e receita](../reports/figures/top_cidades_vendas_receita.png)

Ribeirão Preto e Cascavel ocupam, respectivamente, a primeira e a segunda posições nos dois rankings.

A partir da terceira posição, os resultados divergem: Maringá ocupa o terceiro lugar em número de vendas, enquanto Blumenau assume essa posição em receita. Isso demonstra que o volume de transações não determina sozinho o desempenho financeiro de cada cidade.
